# Sports League Scheduler

**Course:** Introduction to AI  
**Project:** Sports League Scheduler  
**League Size:** 16 teams (Algerian Ligue 1)

## Problem Formulation

We address the double round-robin scheduling problem for a professional sports league.

**Input:**
- n teams (n even, 16 in our main experiment)
- GPS coordinates of each team's stadium
- List of derby matches (rivalries requiring weekend slots)

**Output:**
- A schedule with 2(n-1) rounds
- Each round contains n/2 matches
- Each match specifies: home team, away team, and day (Friday, Saturday, or weekday)

**Hard Constraints (must be satisfied):**
- Each team plays exactly once per round
- Every pair of teams meets exactly twice (once home, once away)

**Soft Constraints (penalized if violated):**
- No team plays more than 2 consecutive away matches (penalty = 500)
- Derby matches not played on Friday or Saturday (penalty = 500)
- Less than 5 days rest between matches (penalty = 500)

**Objective:**
- Minimize total travel distance + sum of all penalties



In [27]:
from __future__ import annotations
import sys
import os
import time
import random
import copy
import csv
import math
from itertools import permutations
from typing import Dict, FrozenSet, List, Optional, Set, Tuple
from collections import deque
from copy import deepcopy
# THOSE THREE FOR THE LAST PROVIDED CHARTS
import streamlit as st
import plotly.express as px
import pandas as pd




__file__ = 'notebook'

# Also fix sys.path if needed
sys.path.insert(0, os.path.abspath('.'))


# Set random seed
random.seed(42)

##  Imports — What and Why

This block loads all the Python libraries the project depends on:

| Import | Purpose |
|--------|---------|
| `sys`, `os` | File path manipulation and locating CSV data files |
| `time` | Measuring how long each algorithm takes to run |
| `random` | Generating random neighbours during search; reproducible via `random.seed(42)` |
| `copy` / `deepcopy` | Safely cloning schedules so the original is never mutated |
| `csv` | Reading `teams.csv` and `derbies.csv` |
| `math` | Computing the Haversine (great-circle) distance between stadiums |
| `itertools.permutations` | Trying all day orderings in the CSP day-assignment step |
| `typing` hints | `Dict`, `List`, `Set`, `FrozenSet`, etc. — document what each function expects |
| `collections.deque` | A fixed-size sliding window for the Tabu list in Hill Climbing |

`random.seed(42)` ensures every run produces identical results, making experiments reproducible.

# Data Strutures

Four lightweight classes model the entire league:

```
Team ──owns──► name, latitude, longitude
Match ────────► home (Team), away (Team), day (str)
Round ────────► round_index, [Match], available days
Schedule ─────► [Round], [Team]
```

The day system rotates on a 3-day week:
- Every round always has **Friday** and **Saturday**.
- The **third day** cycles through `["Wednesday", "Thursday", "Sunday"]` based on `round_index % 3`.

**Quick example:**
```python
algiers = Team("MC Alger", lat=36.74, lon=3.06)
oran    = Team("MC Oran",  lat=35.69, lon=0.63)
m = Match(home=algiers, away=oran, day="Saturday")

print(m)           
print(m.pair())   

`pair()` returns a `frozenset` so `{A, B} == {B, A}` — useful when checking derby matchups regardless of who is home.

In [28]:

THIRD_DAY_OPTION = ["Wednesday", "Thursday", "Sunday"]
ROUND_DAYS = ["Friday", "Saturday"]

class Team:
    def __init__ (self, name, lat, lon):
        self.name = name
        self.lat = lat
        self.lon = lon

    def repr(self):
        return f"Team({self.name})"
class Match:
    def __init__ ( self , home , away , day = None ) :
        self.home = home 
        self.away = away 
        self.day = day 

    def __repr__ (self) :
        return f"{self.home.name} (H) vs {self.away.name} | {self.day} "

    def pair ( self ) :
        return frozenset ( {self.home.name , self.away.name })
    
    def ordered_pair ( self ) : 
        return ( self.home.name , self.away.name , self.day  )
class Round:
    def __init__ ( self , round_index , matches = None ):
        self.round_index = round_index 
        self.matches = matches if matches is not None else []
        self.third_day = THIRD_DAY_OPTION [ round_index % len ( THIRD_DAY_OPTION ) ]
        self.days = ROUND_DAYS + [self.third_day]
        
    def __repr__ ( self ):
        return f" Round {self.round_index + 1 } | Days : {self.days} | Matches : {len(self.matches)}"
        
class Schedule:
    def __init__ ( self , rounds = None , teams = None):
        self.rounds = rounds if rounds is not None else []
        self.teams = teams if teams is not None else []
        
    def total_rounds( self ) : 
        return len( self.rounds )
    
    def total_matches ( self ):
        return sum( len(r.matches) for r in self.rounds )
    
    def print_schedule ( self ) : 
        for r in self.rounds:
            print(f"\n--- Round {r.round_index + 1} | Days: {r.days} ---")
            for match in r.matches:
                print(f"  {match}")
                



# DATA LOADER :

This section reads the two CSV files and builds the distance matrix used by the cost function.

### `load_teams(filepath)` → `list[Team]`
Reads `teams.csv` (columns: `team`, `latitude`, `longitude`) and returns a list of `Team` objects.

### `haversine(lat1, lon1, lat2, lon2)` → `float` (km)
Computes the great-circle distance between two GPS points using the **Haversine formula** — standard for distances on a sphere.

**Example:**
```python
d = haversine(36.74, 3.06, 35.69, 0.63)   
```

### `build_distance_matrix(teams)` → `dict[str, dict[str, float]]`
Builds a symmetric lookup table so `dist["MC Alger"]["MC Oran"]` returns 337 km in O(1).
Each pair is computed once; the symmetric entry is filled automatically.

**Example:**
```python
teams, dist = load_all("data/teams.csv")
print(dist["MC Alger"]["USM Alger"])       
print(dist["JS Saoura"]["CS Constantine"])   
```

### `verify_distances(dist_matrix)`
Prints a sanity-check table comparing computed distances against known approximate values — useful for catching coordinate errors early.


In [29]:

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))

def load_derbies(file_path):
    derbies = set()
    with open(file_path, "r", encoding="latin-1") as f:
        reader = csv.DictReader(f)

        for row in reader:
            team_a = row["team_a"]
            team_b = row["team_b"]

            derbies.add(frozenset([team_a, team_b]))

    return derbies


# Load teams
def load_teams(filepath: str) -> list:
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")

    teams = []
    with open(filepath, newline="", encoding="latin-1") as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = row["team"].strip()
            teams.append(Team(
                name         = name,
                lat          = float(row["latitude"]),
                lon          = float(row["longitude"])
                ))
    return teams

# Haversine formula
_EARTH_RADIUS_KM = 6_371.0
def haversine(lat1: float, lon1: float,
              lat2: float, lon2: float) -> float:
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)

    a = (math.sin(dphi / 2) ** 2
         + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return round(_EARTH_RADIUS_KM * c, 2)


# Dict-of-dicts distance matrix  
def build_distance_matrix(teams: list) -> dict:
    dist = {t.name: {} for t in teams}

    for a in teams:
        for b in teams:
            if a.name == b.name:
                dist[a.name][b.name] = 0.0
            elif b.name not in dist[a.name]:
                d = haversine(a.lat, a.lon, b.lat, b.lon)
                dist[a.name][b.name] = d
                dist[b.name][a.name] = d

    return dist



# Convenience wrapper

def load_all(filepath: str) -> tuple:
    teams       = load_teams(filepath)
    dist_matrix = build_distance_matrix(teams)
    return teams, dist_matrix


def verify_distances(dist_matrix: dict) -> None:
    checks = [
        ("MC Alger",  "USM Alger",      "~9 km (same city)"),
        ("MC Alger",  "MC Oran",        "~337 km"),
        ("MC Alger",  "CS Constantine", "~328 km"),
        ("MC Alger",  "JS Saoura",      "~739 km"),
        ("JS Saoura", "CS Constantine", "~971 km"),
    ]
    print("\n=== Distance Verification ===")
    for a, b, note in checks:
        d = dist_matrix.get(a, {}).get(b, "NOT FOUND")
        print(f"  {a:<22} → {b:<22}: {d!s:>8} km   (expected {note})")
    print()



# SCHEDULE GENERATOR :

### `double_round_robin(teams)` → `Schedule`
Generates a complete double round-robin schedule using the **circle (polygon) rotation method**:

1. One team is fixed as the "anchor" (`teams[0]`).
2. The remaining N−1 teams rotate one position clockwise each round.
3. This produces N−1 rounds for the **first leg** (each pair plays once).
4. The **second leg** mirrors the first with home/away roles swapped.

**Result for N teams:**
- `2*(N-1)` total rounds
- `N/2` matches per round
- `N*(N-1)` total matches

**Example with 4 teams [A, B, C, D]:**
```
Round 1:  A vs D,  B vs C
Round 2:  A vs C,  D vs B
Round 3:  A vs B,  C vs D
(second leg swaps home/away for all matches)
```

### `assign_all_days` / `assign_match_day(round_obj)`
Distributes matches across the three available days using a simple heuristic:
- **First match** → Saturday
- **Last two matches** → Friday
- **Middle matches** → the rotating third day (Wed/Thu/Sun)


In [30]:
def double_round_robin ( teams ) : 
    n = len( teams )
    if n % 2 != 0:
        raise ValueError(f"Need an even number of teams, got {n}")
    if n < 2:
        raise ValueError(f"Need at least 2 teams, got {n}")

    fixed = teams [ 0 ]
    rotating = list ( teams[1:] )
    
    first_leg = []
    for round_num in range ( n - 1 ):
        round_obj = Round ( round_index = round_num )
        if round_num % 2 == 0 : 
            round_obj.matches.append ( Match ( home = fixed , away = rotating [ 0 ]))
        else:
            round_obj.matches.append ( Match ( home = rotating [ 0 ] , away = fixed ))
        
        for i in range ( 1 , n // 2 ):
            team_a = rotating [ i ]
            team_b = rotating [ n - 1 - i ]
            if i % 2 == 0 :
                round_obj.matches.append ( Match ( home = team_a , away = team_b ))
            else : 
                round_obj.matches.append ( Match ( home = team_b , away = team_a ))
                
        first_leg.append(round_obj)
        
        last = rotating.pop()
        rotating.insert ( 0 , last )
        
    sec_leg = []
    for i , round_obj in enumerate(first_leg):
        mirror = Round ( round_index= (n-1) + i)
        for match in round_obj.matches : 
            mirror.matches.append( Match ( home = match.away , away = match.home ))
        
        sec_leg.append ( mirror )
        
    all_rounds = first_leg + sec_leg
    
    return Schedule ( rounds = all_rounds , teams= list ( teams ))

def assign_all_days ( schedule ) : 
    for round_obj in schedule.rounds : 
        assign_match_day ( round_obj )
        
    return schedule

def assign_match_day ( round_obj ) : 
    matches = round_obj.matches
    n = len ( matches )
    saturday_matches = matches [:1]
    friday_matches = matches[n-2:]
    third_matches = matches[1:n-2]
    
    for match in saturday_matches : 
        match.day = "Saturday"
    for match in friday_matches : 
        match.day = "Friday"
    for match in third_matches : 
        match.day = round_obj.third_day 


# NEIGHBOURS :

These functions define the **moves** available to every local search algorithm. Each one returns a **deep copy** — the original schedule is never modified.

| Function | What changes | Example |
|----------|-------------|---------|
| `swap_rounds(s, i, j)` | Swaps match sets of rounds i and j | Round 1 ↔ Round 5 |
| `swap_home_away(s, round_i, match_i)` | Flips home/away for a pair across **both** legs | A(H) vs B → B(H) vs A, mirror updated |
| `swap_days_in_round(s, round_i, a, b)` | Exchanges the day of two matches within a round | Match 0: Fri ↔ Match 2: Sun |
| `swap_third_day(s, i, j)` | Exchanges the third day between two rounds | Round 1's "Wed" ↔ Round 3's "Sun" |

**Why deep copy?** Each neighbour is an independent candidate — modifying in place would corrupt the current solution.


### `generate_one_random_neighbor(schedule)`
Picks one move type at random and applies it to a random location. Used by SA and stochastic Hill Climbing .


In [31]:

def swap_rounds ( schedule , round_i , round_j ) : 
    new_schedule = copy.deepcopy ( schedule )
    
    new_schedule.rounds[round_i].matches , new_schedule.rounds[round_j].matches = new_schedule.rounds[round_j].matches , new_schedule.rounds[round_i].matches
    
    new_schedule.rounds[round_i].round_index = round_i
    new_schedule.rounds[round_j].round_index = round_j
    
    return new_schedule


def swap_home_away(schedule, round_i, match_i):
    new_schedule = copy.deepcopy(schedule)
    
    match = new_schedule.rounds[round_i].matches[match_i]
    pair  = match.pair()  
    
    match.home, match.away = match.away, match.home
    

    for r_idx, round_obj in enumerate(new_schedule.rounds):
        if r_idx == round_i:
            continue
        for m in round_obj.matches:
            if m.pair() == pair and m.home.name == match.away.name:
                m.home, m.away = m.away, m.home
                return new_schedule
    match.home, match.away = match.away, match.home
    return new_schedule

def swap_days_in_round ( schedule , round_i , match_i_a , match_i_b ):
    new_schedule = copy.deepcopy ( schedule )
    round_obj = new_schedule.rounds[round_i]
    
    round_obj.matches[match_i_a].day , round_obj.matches[match_i_b].day = round_obj.matches[match_i_b].day , round_obj.matches[match_i_a].day
    
    return new_schedule

def swap_third_day ( schedule , round_i , round_j ) : 
    new_schedule = copy.deepcopy ( schedule )
    
    round_a = new_schedule.rounds [ round_i ]
    round_b = new_schedule.rounds [ round_j ]
    
    old_third_a = round_a.third_day
    old_third_b = round_b.third_day
    
    round_a.third_day , round_b.third_day = old_third_b , old_third_a
    
    round_a.days = ["Friday", "Saturday", round_a.third_day]
    round_b.days = ["Friday", "Saturday", round_b.third_day]
    
    for match in round_a.matches : 
        if match.day == old_third_a : 
            match.day = round_a.third_day
               
    for match in round_b.matches : 
        if match.day == old_third_b : 
            match.day = round_b.third_day
             
        
    return new_schedule

               
def generate_one_random_neighbor(schedule):
    n = len(schedule.rounds)
    move = random.choice([1, 2, 3, 4])

    if move == 1:
        i, j = random.sample(range(n), 2)
        return swap_rounds(schedule, i, j)

    elif move == 2:
        i = random.randint(0, n - 1)
        m_idx = random.randint(0, len(schedule.rounds[i].matches) - 1)
        return swap_home_away(schedule, i, m_idx)

    elif move == 3:
        i = random.randint(0, n - 1)
        n_matches = len(schedule.rounds[i].matches)
        a, b = random.sample(range(n_matches), 2)
        return swap_days_in_round (schedule, i, a, b)

    elif move == 4:
        i, j = random.sample(range(n), 2)
        return swap_third_day(schedule, i, j)



# HARD CINSTRAINTS : 

These functions verify whether a generated schedule satisfies all mandatory championship constraints.

A schedule is considered valid only if all hard constraints are respected.

---

## `check_one_match_per_round(schedule)`

Verifies that each team plays exactly once per round.

The function:

- tracks which teams already appeared in each round
- checks for duplicate appearances

### Returns

- `False` if any team appears twice in the same round
- otherwise `True`

---

## `check_all_pairs_exactly_twice(schedule)`

Counts how many times each unordered pair of teams plays against each other.

Checks that:

- every pair of teams plays exactly two matches
- one home match
- one away match

This enforces the double round-robin structure.

### Returns

- `False` if any pair does not play exactly twice
- otherwise `True`

---

## `check_home_away_balance(schedule)`

Counts the number of:

- home matches
- away matches

for every team.

Checks that the difference between home and away games is less than `1`.

### Returns

- `False` if a team has an imbalanced schedule
- otherwise `True`

Example of invalid case:

- a team plays `10` home games and `7` away games

---

## `is_partially_valid(schedule)`

Performs a lightweight partial validation used during CSP search.

Checks only:

- no team plays twice in the same round
- no pair plays more than twice

This improves efficiency during backtracking by avoiding expensive full validation at every step.

### Returns

- `True` if the partial schedule is still feasible
- otherwise `False`

---

## `is_valid(schedule)`

Performs complete validation of the schedule.

Combines all hard constraints:

- one match per round
- each pair plays exactly twice
- balanced home/away games

### Returns

- `True` only if all constraints are satisfied
- otherwise `False`

---

## `csp_validate(schedule)`

Provides detailed validation feedback instead of a simple boolean result.

Instead of returning only `True` or `False`, the function returns:

- a list of violation messages
- explanations of which constraints were broken

This is useful for:

- debugging
- CSP analysis
- understanding invalid schedules

In [32]:

def check_one_match_per_round(schedule):
    for r in schedule.rounds:
        teams_seen = set()
        for match in r.matches:
            if match is None:
                continue
            t1 = match.home.name
            t2 = match.away.name
            if t1 in teams_seen or t2 in teams_seen:
                return False
            teams_seen.add(t1)
            teams_seen.add(t2)
    return True


def check_all_pairs_exactly_twice(schedule):
    pair_count = {}
    
    for r in schedule.rounds:
        for match in r.matches:
            if match is None:
                continue
            pair = frozenset({match.home.name, match.away.name})   
            pair_count[pair] = pair_count.get(pair, 0) + 1
    
    teams = [t.name for t in schedule.teams]
    
    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            pair = frozenset({teams[i], teams[j]})
            if pair_count.get(pair, 0) != 2:
                return False
    
    return True


def check_home_away_balance(schedule):
    home_count = {}
    away_count = {}
    
    for r in schedule.rounds:
        for match in r.matches:
            if match is None:
                continue
            home = match.home.name
            away = match.away.name
            home_count[home] = home_count.get(home, 0) + 1
            away_count[away] = away_count.get(away, 0) + 1
    
    for team in schedule.teams:
        home = home_count.get(team.name, 0)
        away = away_count.get(team.name, 0)
        if abs(home - away) >= 1:
            return False
    
    return True


def is_partially_valid(schedule):
    # only checks that no team plays more than once per round and that no pair of teams plays more than twice
    for r in schedule.rounds:
        teams_seen = set()
        for m in r.matches:
            if m is None:
                continue
            if m.home.name in teams_seen or m.away.name in teams_seen:
                return False
            teams_seen.add(m.home.name)
            teams_seen.add(m.away.name)
    
    pair_count = {}
    for r in schedule.rounds:
        for m in r.matches:
            if m is None:
                continue
            pair = frozenset({m.home.name, m.away.name})
            pair_count[pair] = pair_count.get(pair, 0) + 1
            if pair_count[pair] > 2:
                return False
    
    return True


def is_valid(schedule):
    return (
        check_one_match_per_round(schedule) and
        check_all_pairs_exactly_twice(schedule) and
        check_home_away_balance(schedule)
    )


def csp_validate(schedule):
    violations = []
    
    for round_index, r in enumerate(schedule.rounds, 1):
        teams_seen = set()
        for match in r.matches:
            if match is None:
                continue
            t1 = match.home.name
            t2 = match.away.name
            
            if t1 in teams_seen:
                violations.append(f"Round {round_index}: Team {t1} plays more than one match.")
            else:
                teams_seen.add(t1)
                
            if t2 in teams_seen:
                violations.append(f"Round {round_index}: Team {t2} plays more than one match.")
            else:
                teams_seen.add(t2)  
    
    pair_counts = {}
    for r in schedule.rounds:
        for match in r.matches:
            if match is None:
                continue
            pair = frozenset({match.home.name, match.away.name})
            pair_counts[pair] = pair_counts.get(pair, 0) + 1
    
    teams = [t.name for t in schedule.teams]
    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            pair = frozenset({teams[i], teams[j]})
            count = pair_counts.get(pair, 0)
            if count != 2:
                violations.append(f"Pair {pair} plays {count} times instead of 2.")
    
    home_count = {}
    away_count = {}
    for r in schedule.rounds:
        for match in r.matches:
            if match is None:
                continue
            home_count[match.home.name] = home_count.get(match.home.name, 0) + 1
            away_count[match.away.name] = away_count.get(match.away.name, 0) + 1
    
    for team in schedule.teams:
        home = home_count.get(team.name, 0)
        away = away_count.get(team.name, 0)
        if abs(home - away) >= 1:
            violations.append(f"Team '{team.name}' has unbalanced home/away: plays {home} home vs {away} away")
    
    return violations

# SOFT CONSTRAINTS :

##  Soft Constraints

Soft constraints are preferences that improve the quality and realism of the schedule.  
Violating them does not make the schedule invalid, but adds penalties to the total cost.

---

##  Derbies Penalty Function

Checks whether derby matches are scheduled outside weekends.

The function:

- iterates over each match in the schedule
- checks if the match is a derby
- verifies whether it is played on a weekend
- adds a penalty if the condition is violated

### Penalty

```python
penalty = 600
```

### Example

- `USM Alger vs MC Alger` scheduled on a Tuesday → `+600`

---

##  Rest Day Penalty

Ensures fairness between teams regarding recovery and rest days.

The function checks that:

- a team does not play again in less than **5 days**
- similar to rest constraints used in normal championship leagues

If this criterion is violated, a penalty is added.

### Penalty

```python
penalty = 300
```

### Example

- a team plays on Monday and again on Thursday → `+300`

---

## Successive Away Penalty

Checks whether a team plays too many consecutive away matches.

If a team plays more than **two away matches in a row**, a penalty is applied.

### Penalty

```python
penalty = 900
```

### Example

- `MC Oran` plays away in rounds `3`, `4`, and `5` → `+900`

In [33]:

def Derbies_penalty(schedule, derbies, weekend, penalty=500):

    total = 0

    for round_obj in schedule.rounds:
        for match in round_obj.matches:

            match_pair = match.pair()

            if match_pair in derbies and match.day not in weekend:
                total += penalty

    return total


def Rest_day_penalty(schedule, teams, penalty=500):
    match_days = {"Friday": 0, "Saturday": 1, "Sunday": 2,"Wednesday":3, "Thursday": 4}
    total = 0
    for team in teams:
        team_match_day = []
        for round_obj in schedule.rounds:
            for match in round_obj.matches:
                if team.name == match.home.name or team.name == match.away.name:  
                    team_match_day.append(match.day)

        for i in range(len(team_match_day) - 1):
            day_i = match_days[team_match_day[i]]
            day_j = match_days[team_match_day[i + 1]]
            if day_j > day_i:
                rest = day_j - day_i - 1
            else:
                rest = (7 - day_i) + day_j - 1
            if rest < 5:
                total += penalty

    return total

def successive_away_penalty(team, schedule, penalty=500):
    total = 0
    counter = 0
    for round_obj in schedule.rounds:
        for match in round_obj.matches:
            if team.name == match.away.name:       
                counter += 1
            elif team.name == match.home.name:     
                counter = 0
            if counter >= 3:
                total += penalty
                counter = 0
    return total



# OBJECTIVE FUNCTION :

The objective function is used to evaluate the quality of a generated schedule.

It combines:

- total travel distance
- soft constraint penalties

This allows fair comparison between all algorithms used in the project.

---

## Total Distance

The function calculates the total distance traveled by each team in the schedule.

For every match:

- the starting point is considered the team's current location
- the destination is the opponent’s home stadium

Initially, every team starts from its own home.

### Travel Continuity

If:

- Team `A` travels to Team `B`
- then directly travels to Team `C`

the second trip is calculated starting from Team `B`’s location, not from Team `A`’s home.

So the current position of the team changes after every away match.

---

### Purpose

By iterating over all matches in the schedule, the function computes the total traveled distance in order to compare algorithms and determine which one produces schedules with lower travel cost.

---

## Cost Function

The cost function serves as the main evaluator and comparator between all algorithms used in the project.

It combines:

- the total travel distance
- penalties generated from soft constraint violations

This creates a balanced evaluation that measures:

- travel efficiency
- soft constraint satisfaction

---

### Algorithms Compared

The cost function is used to compare the performance of:

- Simulated Annealing (SA)
- Hill Climbing (HC)
- Greedy Search
- CSP

in terms of both optimization quality and realism of the produced schedule.

In [34]:
def total_distance(schedule, dist_Matric, teams):
    total = 0
    current_position = {team.name: team.name for team in teams}

    for round_obj in schedule.rounds:
        for match in round_obj.matches:
            from_city = current_position[match.away.name]
            to_city = match.home.name

            total += dist_Matric[from_city][ to_city]

            current_position[match.away.name] = match.home.name
            

    return total



def cost_function(schedule, teams, dict_Matric, derbies, weekend,penalty=500 ):

    total_cost = 0

    total_cost += total_distance(schedule, dict_Matric, teams)
    total_cost += Rest_day_penalty(schedule, teams,penalty)
    total_cost += Derbies_penalty(schedule, derbies, weekend,penalty)
    for team in teams:
        total_cost+=successive_away_penalty(team, schedule,penalty)

    
    return total_cost


# SEARCH ALGORITHMS :

## GREEDY :
The greedy algorithm builds a good schedule in two sequential phases with no backtracking.

### Phase 1 — Round Ordering (`_greedy_round_order`)
Starting from the initial round-robin, the algorithm greedily picks the **next best round** to schedule:

1. For each remaining round, estimate its cost if played next (travel km + away-streak penalty).
2. Lock in the round with the lowest estimated cost.
3. Repeat until all 30 rounds are ordered.

After ordering, a **2-opt refinement** swaps pairs of rounds to check if reversing any sub-sequence improves the total score.

**Intuition:** if two teams are both near Oran this week, scheduling an Oran home game now avoids long detours later.

### Phase 2 — Day Assignment (`_greedy_assign_days`)
For each round, assigns each match to the best available day:
- **Derby matches** are prioritised for weekend slots.
- **Regular matches** fill the remaining slots.
- The chosen day minimises the combined rest-day + consecutive-away penalty.

### `greedy_schedule(...)` → `(Schedule, float, list)`
Top-level entry point. Returns the best schedule, its cost, and a history list of costs (one per committed round) useful for plotting convergence.

```python
best, cost, history = greedy_schedule(
    initial_schedule, derbies, WEEKEND_DAYS, teams, distances
)
print(f"Greedy cost: {cost:,.0f}")
```


In [35]:

DAY_OFFSET = {
    "Friday": 0, "Saturday": 1, "Sunday": 2,
    "Wednesday": 3, "Thursday": 4,
}
REST_MIN = 6
PENALTY  = 500
WEEKENDS = {"Friday", "Saturday"}


STREAK_WEIGHT = 3000   

def _round_travel_cost(round_obj: Round,
                       prev_location: dict,
                       dist_matrix: dict) -> float:
    total = 0.0
    for match in round_obj.matches:
        if match is None:
            continue
        origin = prev_location.get(match.away.name, match.away.name)
        total += dist_matrix.get(origin, {}).get(match.home.name, 0.0)
    return total


def _apply_positions(round_obj: Round, positions: dict) -> dict:
    updated = dict(positions)
    for match in round_obj.matches:
        if match is None:
            continue
        updated[match.away.name] = match.home.name
        updated[match.home.name] = match.home.name
    return updated


def _away_streak(team_name: str, ordered_so_far: list) -> int:
    """Count consecutive away games for team_name at the tail of ordered_so_far."""
    streak = 0
    for r in reversed(ordered_so_far):
        is_home = any(m.home.name == team_name for m in r.matches)
        is_away = any(m.away.name == team_name for m in r.matches)
        if is_home:
            return streak
        if is_away:
            streak += 1
        else:
            break
    return streak


def _round_score(round_obj: Round,
                 ordered_so_far: list,
                 prev_location: dict,
                 dist_matrix: dict) -> float:
    km = _round_travel_cost(round_obj, prev_location, dist_matrix)

    # count how many teams in this round would extend an existing away streak to ≥ 3
    streak_violations = sum(
        1
        for m in round_obj.matches
        if m is not None and _away_streak(m.away.name, ordered_so_far) >= 2
    )

    return km + STREAK_WEIGHT * streak_violations


def _sequence_score(order: list,
                    rounds: list,
                    dist_matrix: dict,
                    teams: list) -> float:

    positions = {t.name: t.name for t in teams}
    ordered_so_far = []
    total = 0.0

    for idx in order:
        r = rounds[idx]
        total += _round_score(r, ordered_so_far, positions, dist_matrix)
        positions = _apply_positions(r, positions)
        ordered_so_far.append(r)

    return total


# ─────────────────────────────────────────────────────────────────────────────
# phase 2 — round ordering with 2-opt
# ─────────────────────────────────────────────────────────────────────────────

def _greedy_round_order(schedule: Schedule,
                        dist_matrix: dict,
                        teams: list,
                        derbies: set,
                        weekends: set,
                        cost_fn,
                        history: list) -> list:

    rounds         = schedule.rounds
    remaining      = list(range(len(rounds)))
    ordered        = []          # list of int indices  (used for 2-opt & history)
    ordered_rounds = []          # list of Round objects (passed to _away_streak)
    positions      = {t.name: t.name for t in schedule.teams}

    # ── Phase A: greedy construction ─────────────────────────────────────────
    while remaining:
        best_idx = min(
            remaining,
            key=lambda i: _round_score(rounds[i], ordered_rounds, positions, dist_matrix)
        )
        ordered.append(best_idx)
        ordered_rounds.append(rounds[best_idx])          # keep Round objects in sync
        positions = _apply_positions(rounds[best_idx], positions)
        remaining.remove(best_idx)

        # FIX 3: record cost_function on partial schedule after each commit
        partial = Schedule(
            rounds=[copy.deepcopy(rounds[i]) for i in ordered],
            teams=schedule.teams
        )
        assign_all_days(partial)
        history.append(round(cost_fn(partial, teams, dist_matrix, derbies, weekends), 2))

    # ── Phase B: 2-opt refinement (streak-aware) ──────────────────────────────
    best_cost = _sequence_score(ordered, rounds, dist_matrix, schedule.teams)
    improved  = True

    while improved:
        improved = False
        for i in range(len(ordered) - 1):
            for j in range(i + 2, len(ordered)):
                candidate      = ordered[:i] + ordered[i:j+1][::-1] + ordered[j+1:]
                candidate_cost = _sequence_score(candidate, rounds, dist_matrix, schedule.teams)
                if candidate_cost < best_cost - 1e-6:
                    ordered    = candidate
                    best_cost  = candidate_cost
                    improved   = True

    return ordered


def _apply_round_order(schedule: Schedule, order: list) -> Schedule:
    new_schedule = copy.deepcopy(schedule)
    old_rounds   = schedule.rounds
    for new_pos, old_pos in enumerate(order):
        new_round         = Round(round_index=new_pos)
        new_round.matches = copy.deepcopy(old_rounds[old_pos].matches)
        new_schedule.rounds[new_pos] = new_round
    return new_schedule


# ─────────────────────────────────────────────────────────────────────────────
# phase 3 — penalty-aware day assignment (unchanged)
# ─────────────────────────────────────────────────────────────────────────────

def _greedy_assign_days(schedule: Schedule,
                        derbies: set,
                        weekends: set,
                        penalty: int) -> Schedule:
    new_schedule      = copy.deepcopy(schedule)
    last_absolute_day = {}
    away_streak_count = {}

    for round_obj in new_schedule.rounds:
        matches = round_obj.matches
        days    = list(round_obj.days)
        if not matches:
            continue

        def absolute_day(day_label: str) -> int:
            return round_obj.round_index * 7 + DAY_OFFSET.get(day_label, 2)

        def penalty_score(match: Match, day_label: str) -> float:
            score       = 0.0
            current_day = absolute_day(day_label)
            for team in (match.home, match.away):
                prev = last_absolute_day.get(team.name)
                if prev is not None and (current_day - prev) < REST_MIN:
                    score += penalty
            if away_streak_count.get(match.away.name, 0) >= 2:
                score += penalty
            pair = frozenset([match.home.name, match.away.name])
            if pair in derbies and day_label not in weekends:
                score += penalty
            return score

        derby_matches   = [m for m in matches if frozenset([m.home.name, m.away.name]) in derbies]
        regular_matches = [m for m in matches if frozenset([m.home.name, m.away.name]) not in derbies]
        used_days       = []

        def choose_best_day(match: Match, prefer_weekend: bool) -> str:
            weekend_days     = [d for d in days if d in weekends     and d not in used_days]
            non_weekend_days = [d for d in days if d not in weekends and d not in used_days]
            available_days   = [d for d in days if d not in used_days]
            pool = (weekend_days + non_weekend_days) if prefer_weekend else (non_weekend_days + weekend_days)
            if not pool:
                pool = available_days if available_days else days
            best_day = min(pool, key=lambda d: penalty_score(match, d))
            used_days.append(best_day)
            return best_day

        assignment = {}
        for match in derby_matches:
            assignment[id(match)] = choose_best_day(match, prefer_weekend=True)
        for match in regular_matches:
            assignment[id(match)] = choose_best_day(match, prefer_weekend=False)

        for match in matches:
            selected_day = assignment.get(id(match), days[0])
            match.day    = selected_day
            current_day  = absolute_day(selected_day)
            last_absolute_day[match.home.name] = current_day
            last_absolute_day[match.away.name] = current_day
            away_streak_count[match.away.name] = away_streak_count.get(match.away.name, 0) + 1
            away_streak_count[match.home.name] = 0

    return new_schedule



def compute_total_travel(schedule: Schedule, dist_matrix: dict) -> float:
    return round(total_distance(schedule, dist_matrix, schedule.teams), 2)


def validate_schedule(schedule: Schedule) -> dict:
    teams  = [t.name for t in schedule.teams]
    n      = len(teams)
    issues = {
        "wrong_round_count": 0, "wrong_matches_per_round": 0,
        "team_appears_twice_in_round": 0, "pair_not_exactly_once": 0,
    }
    if schedule.total_rounds() != (n - 1) * 2:
        issues["wrong_round_count"] = abs(schedule.total_rounds() - (n - 1) * 2)
    pair_count = {}
    for r in schedule.rounds:
        if len(r.matches) != n // 2:
            issues["wrong_matches_per_round"] += 1
        seen = set()
        for m in r.matches:
            if m.home.name in seen or m.away.name in seen:
                issues["team_appears_twice_in_round"] += 1
            seen.add(m.home.name); seen.add(m.away.name)
            key = (m.home.name, m.away.name)
            pair_count[key] = pair_count.get(key, 0) + 1
    for t1 in teams:
        for t2 in teams:
            if t1 != t2 and pair_count.get((t1, t2), 0) != 1:
                issues["pair_not_exactly_once"] += 1
    return issues


def greedy_schedule(schedule,
                    derbies: set,
                    weekends: set,
                    teams: list,
                    dist_matrix: dict,
                    seed: int = 42,
                    verbose: bool = False,
                    cost_fn=None) -> tuple:
    """
    Returns (schedule, best_cost, history) — same signature as SA.

    history[i] = cost_function value after the (i+1)-th round was committed.
    This gives ~30 history points, directly comparable to SA's per-iteration
    history on the same y-axis.
    """
    if cost_fn is None:
        cost_fn = cost_function

    random.seed(seed)
    t0      = time.time()
    history = []

    raw   = double_round_robin(teams)
    base  = assign_all_days(raw)

    # Phase 2: streak-aware round ordering (history filled inside)
    order            = _greedy_round_order(base, dist_matrix, teams,
                                           derbies, weekends, cost_fn, history)
    ordered_schedule = _apply_round_order(base, order)

    # Phase 3: penalty-aware day assignment
    final_schedule = _greedy_assign_days(ordered_schedule, derbies, WEEKENDS, PENALTY)
    best_cost      = cost_fn(final_schedule, teams, dist_matrix, derbies, weekends)

    # overwrite last history point with true final cost (days reassigned in phase 3)
    if history:
        history[-1] = round(best_cost, 2)

    if verbose:
        elapsed = time.time() - t0
        print(f"[greedy] rounds={final_schedule.total_rounds()} | "
              f"travel={best_cost:,.1f} | time={elapsed:.4f}s")

    return final_schedule, best_cost, history


 #### Hill Climbing with Random Restarts & Tabu Memory

- Samples 50 random neighbors per step (n_samples=50).
- Allows up to 10 sideways moves before triggering a restart (max_sideways=10).
- Keeps a tabu list of 20 states to prevent cycling (tabu_size=20).
- Runs up to 5 random restarts, each capped at 200 iterations (max_restarts=5, max_iterations=200).
- Applies a penalty of 500 per constraint violation (penalty=500).

---

#### Sideways Moves
- Accepts equal-cost neighbors when no improvement is found, to escape flat regions.
- Counter resets to zero on every improvement, restoring the full sideways budget.
- Once the limit of 10 is reached, a random restart is triggered immediately.

#### Tabu List
- Each visited state is fingerprinted as a tuple of ordered match pairs.
- Any neighbor matching a stored fingerprint is skipped, preventing oscillation.
- Oldest entry is evicted automatically once the queue reaches 20 states.
#### How it works 
1. Sample — 50 random neighbors are generated from the current schedule.
2. Classify — each neighbor is labeled as *improving*, *sideways*, or *worsening*.
3. Move — best improving neighbor is chosen; if none, best non-tabu sideways is taken.
4. Tabu update — current state fingerprint is added to the tabu queue before moving.
5. Best update — if the new cost beats the global best, it is saved immediately.
6. Stuck check — if no improving or sideways move exists, a restart is triggered.

#### Random restarts
After hitting the sideways or stuck limit, the team list is shuffled and a fresh round-robin is generated. The **global best** is preserved across all restarts.






In [36]:

def print_solution_details(schedule, teams, distances, derbies, weekends,
                           penalty=500, label=""):
    dist  = total_distance(schedule, distances, teams)
    rest  = Rest_day_penalty(schedule, teams, penalty)
    derby = Derbies_penalty(schedule, derbies, weekends, penalty)

    away_total = 0
    away_violations = []
    for team in teams:
        v = successive_away_penalty(team, schedule, penalty)
        if v > 0:
            away_violations.append((team.name, v // penalty, v))
            away_total += v

    total = dist + rest + derby + away_total

    title = f"  {label}  " if label else "  Schedule Details  "
    print("\n" + "═" * 52)
    print(f"{title:^52}")
    print("═" * 52)
    print(f"  {'Total travel distance':<30} {dist:>10,.1f} km")
    print("─" * 52)
    print(f"  {'Rest day violations':<30} {rest // penalty:>10}  ×{penalty}")
    print(f"  {'  → penalty cost':<30} {rest:>10,.0f}")
    print("─" * 52)
    print(f"  {'Derby not on weekend':<30} {derby // penalty:>10}  ×{penalty}")
    print(f"  {'  → penalty cost':<30} {derby:>10,.0f}")
    print("─" * 52)
    if away_violations:
        print(f"  Successive away (≥3) violations:")
        for name, count, cost in away_violations:
            print(f"    {name:<26} {count:>2} violation(s)  +{cost:,.0f}")
        print(f"  {'  → penalty cost':<30} {away_total:>10,.0f}")
    else:
        print(f"  {'Successive away violations':<30} {'none':>10}")
    print("═" * 52)
    print(f"  {'GRAND TOTAL COST':<30} {total:>10,.1f}")
    print("═" * 52 + "\n")

    return {
        "distance"        : dist,
        "rest_violations" : rest  // penalty,
        "rest_penalty"    : rest,
        "derby_violations": derby // penalty,
        "derby_penalty"   : derby,
        "away_violations" : len(away_violations),
        "away_penalty"    : away_total,
        "total"           : total,
    }


# ── Tabu helpers ──────────────────────────────────────────

def _schedule_fingerprint(schedule) -> tuple:
    return tuple(
        m.ordered_pair()
        for r in schedule.rounds
        for m in r.matches
    )


def _best_sideways(candidates, distances, teams, tabu):
    best_nb, best_dist = None, float("inf")
    for nb, _cost in candidates:
        fp = _schedule_fingerprint(nb)
        if fp in tabu:
            continue
        d = total_distance(nb, distances, teams)
        if d < best_dist:
            best_nb, best_dist = nb, d
    return best_nb  # None if every candidate was tabu


#  Stochastic step 

def _stochastic_step(current, current_cost, cost_fn, teams, distances, derbies, weekends,
                     n_samples=50, tabu=None):
  
    if tabu is None:
        tabu = set()

    improving           = []
    sideways_candidates = []
    for _ in range(n_samples):
        nb = generate_one_random_neighbor(current)
        c  = cost_fn(nb, teams, distances, derbies, weekends)
        if c < current_cost:
            improving.append((nb, c))
        elif c == current_cost:
            sideways_candidates.append((nb, c))

    if improving:
        best = min(improving, key=lambda x: x[1])
        return best[0], best[1], "improved"

    chosen = _best_sideways(sideways_candidates, distances, teams, tabu)
    if chosen is not None:
        return chosen, current_cost, "sideways"

    return current, current_cost, "stuck"


# ── Random-restart helper ─────────────────────────────────────────────────────

def _random_restart_schedule(teams):
    shuffled = list(teams)
    random.shuffle(shuffled)
    return assign_all_days(double_round_robin(shuffled))


# ── Main entry point ──────────────────────────────────────────────────────────

def hill_climbing(
    schedule,
    distances,
    derbies,
    weekends,
    teams,
    penalty        = 500,
    max_restarts   = 5,
    max_iterations = 200,
    n_samples      = 50,   # neighbours sampled per step
    max_sideways   = 10,   # max consecutive sideways moves before restart
    tabu_size      = 20,   # sliding-window tabu size
    cost_fn        = None,
    verbose        = True,
):
    if cost_fn is None:
        cost_fn = cost_function

    best      = deepcopy(schedule)
    best_cost = cost_fn(best, teams, distances, derbies, weekends)
    history   = []
    global_step = 0

    for restart in range(max_restarts + 1):
        current = (
            deepcopy(schedule) if restart == 0
            else _random_restart_schedule(teams)
        )
        current_cost  = cost_fn(current, teams, distances, derbies, weekends)
        sideways_used = 0

        # FIFO tabu window: deque for ordered eviction, set for O(1) lookup
        tabu_q = deque(maxlen=tabu_size)
        tabu_s = set()

        if verbose:
            tag = "initial" if restart == 0 else f"restart {restart}"
            print(f"\n  ── stochastic  {tag}  (start cost={current_cost:,.1f})")

        for it in range(max_iterations):
            next_s, next_cost, status = _stochastic_step(
                current, current_cost,
                cost_fn, teams, distances, derbies, weekends,
                n_samples=n_samples, tabu=tabu_s,
            )

            if status == "improved":
                sideways_used = 0
                fp = _schedule_fingerprint(current)
                if fp not in tabu_s:
                    if len(tabu_q) == tabu_size:
                        tabu_s.discard(tabu_q[0])
                    tabu_q.append(fp)
                    tabu_s.add(fp)

                current      = next_s
                current_cost = next_cost
                global_step += 1

                if current_cost < best_cost:
                    best      = deepcopy(current)
                    best_cost = current_cost

                history.append(best_cost)

                if verbose:
                    print(f"    step {global_step:>4}  cost={best_cost:,.1f}")

            elif status == "sideways":
                sideways_used += 1
                if sideways_used > max_sideways:
                    if verbose:
                        print(f"    sideways limit ({max_sideways}) reached after "
                              f"{it + 1} steps — triggering restart")
                    break

                fp = _schedule_fingerprint(current)
                if fp not in tabu_s:
                    if len(tabu_q) == tabu_size:
                        tabu_s.discard(tabu_q[0])
                    tabu_q.append(fp)
                    tabu_s.add(fp)

                current      = next_s
                current_cost = next_cost
                global_step += 1
                history.append(best_cost)

                if verbose:
                    print(f"    step {global_step:>4}  [sideways {sideways_used}/{max_sideways}]"
                          f"  cost={current_cost:,.1f}")

            else:  # "stuck"
                if verbose:
                    print(f"    truly stuck after {it + 1} steps — triggering restart")
                break

    return best, best_cost, history




### Simulated Annealing :

Simulated Annealing (SA) escapes local optima by accepting worse solutions with a probability:

\[
P = e^{-\Delta / T}
\]

which decreases as the "temperature" `T` cools each iteration — allowing it to converge to an inefficient solution at the end.

---

#### How it works

Each iteration:

1. **Initialize all parameters**
   - `current` with the initial schedule from round robin
   - `best` with the initial current
   - `best cost` calculated using the cost function
   - `T` with `T_start`
   - `no_improve_count` and `it` with `0`
   - create the history list in order to track the convergence

2. **Generate one random neighbour**
   ```python
   generate_one_random_neighbor(schedule)
   ```

3. **Compute**
   ```python
   delta = neighbour_cost − current_cost
   ```

4. If `delta < 0` (improvement) → always accept.

5. If `delta ≥ 0` (worse) → accept with probability:

   \[
   P = exp(-\Delta / T)
   \]

   - High `T` (hot) → almost any move is accepted → broad exploration.
   - Low `T` (cold) → only improvements accepted → fine-tuning.

6. **Cool down**
   ```python
   T = T × alpha
   ```

7. append best cost to history list.

---

### Reheat mechanism

If `no_improve_count` reaches `reheat_every` iterations without improvement, `T` is boosted by `reheat_factor` to escape prolonged plateaus — then cooling resumes.

---

### Parameters used in testing

```python
sa_best, sa_cost, sa_history = simulated_annealing(
    schedule=initial_schedule,
    T_start=8000,        # starting temperature
    T_min=0.1,           # stop when this cold
    alpha=0.9997,        # slow cooling → thorough search
    max_iteration=30000,
    reheat_every=4000,   # steps without improvement before reheat
    reheat_factor=2.0,
    cost_fn=cost_function
)
```

---

### Intuition

At `T = 8000`, a move that costs `+500` extra is accepted ~94% of the time.

At `T = 10`, the same move is accepted ~0.000002% of the time — effectively pure hill climbing at that point.

In [37]:

def simulated_annealing(schedule ,distances ,derbies ,weekends,teams,T_start ,T_min ,alpha ,max_iteration, cost_fn=None, reheat_every=2000, reheat_factor=2.0):
    T = T_start
    current=deepcopy(schedule)
    current_cost = cost_fn(current,teams,distances,derbies,weekends)
    best = deepcopy(current)
    best_cost = cost_fn(best,teams,distances,derbies,weekends)
    no_improve_count = 0
    last_best = best_cost
    it=0
    history = [] #track improvment of cost
    while it < max_iteration and T > T_min:

        neighbor = generate_one_random_neighbor(current)
        neighbor_cost = cost_fn(neighbor,teams,distances,derbies,weekends)
        delta = neighbor_cost - current_cost
        if delta < 0:
            current = neighbor
            current_cost = neighbor_cost
        else:
            P = math.exp(-delta/T)
            r=random.random()
            if r< P:
                current = neighbor
                current_cost = neighbor_cost
            else:
                pass
        if current_cost<best_cost:
            best = deepcopy(current)
            best_cost = current_cost
        
      
        if best_cost < last_best:          
            last_best = best_cost
            no_improve_count = 0
        else:
            no_improve_count += 1

        if no_improve_count >= reheat_every:
            T = min(T * reheat_factor, T_start * 0.5)  
            no_improve_count = 0
            last_best = best_cost  
        T = T * alpha
        history.append(best_cost)
        it +=1    
    
    
    return best , best_cost , history    

 

##### CSP SOLVER :

#### Class Initialization

###### `__init__(self, teams, derbies, weekend, dist_matrix)`

Initializes all data structures for the CSP solver.

Creates:

- an empty schedule grid
- tracking for which teams are used in each round
- counters for how many times each pair has played
- monitoring for consecutive away matches
- storage for current positions used in travel distance calculation

---

#### Core CSP Functions

#### `get_possible_matches(self, round_num, match_idx)`

Returns all valid ordered pairs `(home, away)` that can be placed in this slot.

A match is valid if:

- both teams are free in this round
- the pair has not already played twice

Example:

- if team `A` and `B` have not played at all,
  both `(A,B)` and `(B,A)` are returned.

---

#### `assign_match(self, round_num, match_idx, home, away)`

Places a match and updates all data structures.

Updates include:

- marking teams as used in the round
- incrementing pair counters
- updating home/away counts
- tracking consecutive away matches
- adding travel distance to the running total

---

### `unassign_match(self, round_num, match_idx)`

Removes a match during backtracking and recalculates all state from scratch.

This ensures correctness even with complex dependencies such as:

- consecutive away counters
- travel positions
- distance accumulation

---

#### `recalculate_state(self)`

Fully recomputes:

- travel distance
- current positions
- consecutive away counts

from the current schedule grid.

Called during backtracking to restore a consistent state after an assignment is undone.

---

#### Heuristic Functions

#### `get_next_empty_slot(self)`

Implements **MRV (Minimum Remaining Values)**.

Scans all empty slots and returns the one with the fewest possible match options.

This prioritizes the most constrained variable first.

---

#### `lcv_order(self, round_num, match_idx, possibilities)`

Implements **LCV (Least Constraining Value)** with full lookahead.

For each candidate match:

1. temporarily assign the match
2. count how many future slots become impossible or restricted

Matches that preserve more future possibilities are tried first.

---

#### `forward_check(self, round_num, match_idx, home, away)`

Checks all future slots after a tentative assignment.

If any future slot has zero possible matches:

- the assignment is rejected immediately

This prunes dead branches before deep backtracking occurs.

---

#### Search Functions

#### `backtrack(self, start_time, time_limit)`

Main recursive CSP search procedure.

Steps:

1. select the next slot using MRV
2. order values using LCV
3. assign a match
4. perform forward checking
5. recurse recursively

Returns `True` when a complete valid schedule is found.

---

#### `solve(self, time_limit)`

Entry point of the solver.

Responsibilities:

- starts the timer
- runs backtracking
- converts the result to a `Schedule` object
- assigns days to matches
- prints statistics

Returns:

- the final schedule
- or `None` if no solution is found

---

### Day Assignment

#### `assign_days_balanced(schedule, derbies, weekend)`

Assigns days to matches after a valid schedule is found.

Distributes matches across:

- Friday
- Saturday
- a rotating third day:
  - Wednesday
  - Thursday
  - or Sunday

The algorithm tries all permutations and selects the one that places the maximum number of derbies on weekends.

In [38]:

SOFT_W = 500


class ScheduleCSP:
    def __init__(
        self,
        teams,
        derbies: Optional[Set[FrozenSet[str]]] = None,
        weekend: Optional[List[str]] = None,
        dist_matrix: Optional[Dict] = None,
    ):
        self.teams = teams
        self.n = len(teams)
        self.derbies: Set[FrozenSet[str]] = derbies if derbies is not None else set()
        self.weekend: List[str] = weekend if weekend is not None else ["Friday", "Saturday"]
        self.dist_matrix = dist_matrix
        
        self.num_rounds = 2 * (self.n - 1)
        self.matches_per_round = self.n // 2

        self.schedule = [[None] * self.matches_per_round for _ in range(self.num_rounds)]
        self.teams_in_round = [[False] * self.n for _ in range(self.num_rounds)]
        
        self.pair_games = {}
        for i in range(self.n):
            for j in range(i + 1, self.n):
                self.pair_games[(i, j)] = {'home_count': 0, 'away_count': 0}
        
        self.home_count = [0] * self.n
        self.away_count = [0] * self.n
        
        self.consecutive_away = [0] * self.n
        
        self.current_position = [i for i in range(self.n)]
        self.current_travel = 0
        
        self.backtracks = 0
        self.nodes = 0

    def get_distance(self, from_team: int, to_team: int) -> float:
        if self.dist_matrix is None:
            return 0
        from_name = self.teams[from_team].name
        to_name = self.teams[to_team].name
        try:
            return self.dist_matrix[from_name][to_name]
        except (KeyError, TypeError):
            return 0

    def calculate_travel(self, team: int, loc: str, opp: int) -> float:
        if loc == "home":
            return 0
        else:
            from_city = self.current_position[team]
            return self.get_distance(from_city, opp)

    def calculate_away_penalty(self, team: int, loc: str) -> int:
        if loc == "away":
            new_consecutive = self.consecutive_away[team] + 1
            if new_consecutive >= 3:
                return SOFT_W
        return 0

    def is_complete(self):
        for r in range(self.num_rounds):
            for m in range(self.matches_per_round):
                if self.schedule[r][m] is None:
                    return False
        return True

    def get_available_teams(self, round_num):
        available = []
        for t in range(self.n):
            if not self.teams_in_round[round_num][t]:
                available.append(t)
        return available

    def get_possible_matches(self, round_num, match_idx):
        available = self.get_available_teams(round_num)
        possible = []
        
        for i in range(len(available)):
            for j in range(i + 1, len(available)):
                home = available[i]
                away = available[j]
                
                pair_key = (min(home, away), max(home, away))
                pair_info = self.pair_games[pair_key]
                total_games = pair_info['home_count'] + pair_info['away_count']
                
                if total_games >= 2:
                    continue
                
                if pair_info['home_count'] < 1:
                    possible.append((home, away))
                if pair_info['away_count'] < 1:
                    possible.append((away, home))
        
        return possible

    def assign_match(self, round_num, match_idx, home, away):
        travel_home = self.calculate_travel(home, "home", away)
        travel_away = self.calculate_travel(away, "away", home)
        away_penalty_home = self.calculate_away_penalty(home, "home")
        away_penalty_away = self.calculate_away_penalty(away, "away")
        
        self.schedule[round_num][match_idx] = (home, away)
        self.teams_in_round[round_num][home] = True
        self.teams_in_round[round_num][away] = True
        
        pair_key = (min(home, away), max(home, away))
        if home < away:
            self.pair_games[pair_key]['home_count'] += 1
        else:
            self.pair_games[pair_key]['away_count'] += 1
        
        self.home_count[home] += 1
        self.away_count[away] += 1
        
        self.consecutive_away[home] = 0
        self.consecutive_away[away] += 1
        
        self.current_travel += travel_home + travel_away + away_penalty_home + away_penalty_away
        self.current_position[home] = home
        self.current_position[away] = home

    def unassign_match(self, round_num, match_idx):
        home, away = self.schedule[round_num][match_idx]
        
        self.schedule[round_num][match_idx] = None
        self.teams_in_round[round_num][home] = False
        self.teams_in_round[round_num][away] = False
        
        pair_key = (min(home, away), max(home, away))
        if home < away:
            self.pair_games[pair_key]['home_count'] -= 1
        else:
            self.pair_games[pair_key]['away_count'] -= 1
        
        self.home_count[home] -= 1
        self.away_count[away] -= 1
        
        self.recalculate_state()

    def recalculate_state(self):
        positions = [i for i in range(self.n)]
        total = 0
        consec_away = [0] * self.n
        
        for r in range(self.num_rounds):
            for m in range(self.matches_per_round):
                assignment = self.schedule[r][m]
                if assignment is None:
                    continue
                home, away = assignment
                
                if positions[away] != home:
                    total += self.get_distance(positions[away], home)
                    positions[away] = home
                positions[home] = home
                
                consec_away[home] = 0
                consec_away[away] += 1
                if consec_away[away] >= 3:
                    total += SOFT_W
        
        self.current_travel = total
        for i in range(self.n):
            self.current_position[i] = positions[i]
            self.consecutive_away[i] = consec_away[i]

    def get_next_empty_slot(self):
        best_round = -1
        best_match = -1
        min_possibilities = float('inf')
        
        for r in range(self.num_rounds):
            for m in range(self.matches_per_round):
                if self.schedule[r][m] is None:
                    possible = len(self.get_possible_matches(r, m))
                    if possible < min_possibilities:
                        min_possibilities = possible
                        best_round = r
                        best_match = m
                        if min_possibilities == 0:
                            return (best_round, best_match)
        
        if best_round == -1:
            return None
        return (best_round, best_match)

    def forward_check(self, round_num, match_idx, home, away):
        for m in range(match_idx + 1, self.matches_per_round):
            if self.schedule[round_num][m] is None:
                if len(self.get_possible_matches(round_num, m)) == 0:
                    return False
        
        for r in range(round_num + 1, self.num_rounds):
            for m in range(self.matches_per_round):
                if self.schedule[r][m] is None:
                    if len(self.get_possible_matches(r, m)) == 0:
                        return False
        
        return True

    def lcv_order(self, round_num, match_idx, possibilities):
        if len(possibilities) <= 1:
            return possibilities
        
        scored = []
        for home, away in possibilities:
            travel_cost = self.calculate_travel(home, "home", away)
            travel_cost += self.calculate_travel(away, "away", home)
            away_penalty = self.calculate_away_penalty(away, "away")
            constraint_score = travel_cost + away_penalty
            
            self.assign_match(round_num, match_idx, home, away)
            
            # FULL LOOKAHEAD - check ALL remaining rounds
            for r in range(round_num + 1, self.num_rounds):
                for m in range(self.matches_per_round):
                    if self.schedule[r][m] is None:
                        poss = len(self.get_possible_matches(r, m))
                        if poss == 0:
                            constraint_score += 1000
                        elif poss == 1:
                            constraint_score += 100
                        elif poss <= 3:
                            constraint_score += 10
            
            self.unassign_match(round_num, match_idx)
            scored.append(((home, away), constraint_score))
        
        scored.sort(key=lambda x: x[1])
        return [x[0] for x in scored]

    def backtrack(self, start_time, time_limit):
        self.nodes += 1
        
        if time.time() - start_time > time_limit:
            return False
        
        if self.is_complete():
            return True
        
        slot = self.get_next_empty_slot()
        if slot is None:
            return False
        
        round_num, match_idx = slot
        possibilities = self.get_possible_matches(round_num, match_idx)
        
        if not possibilities:
            return False
        
        possibilities = self.lcv_order(round_num, match_idx, possibilities)
        
        for home, away in possibilities:
            self.assign_match(round_num, match_idx, home, away)
            self.backtracks += 1
            
            if self.forward_check(round_num, match_idx, home, away):
                if self.backtrack(start_time, time_limit):
                    return True
            
            self.unassign_match(round_num, match_idx)
        
        return False

    def to_schedule(self):
        rounds = []
        
        for r in range(self.num_rounds):
            matches = []
            for m in range(self.matches_per_round):
                assignment = self.schedule[r][m]
                if assignment is not None:
                    home_idx, away_idx = assignment
                    home_team = self.teams[home_idx]
                    away_team = self.teams[away_idx]
                    match = Match(home=home_team, away=away_team)
                    matches.append(match)
            round_obj = Round(r, matches)
            rounds.append(round_obj)
        
        return Schedule(rounds, self.teams)

    def solve(self, time_limit=60):
        start_time = time.time()
        
        print(f"\n{'='*60}")
        print(f"CSP Solver for {self.n} teams (full lookahead)")
        print(f"Rounds: {self.num_rounds}, Matches per round: {self.matches_per_round}")
        print(f"{'='*60}")
        
        found = self.backtrack(start_time, time_limit)
        elapsed = time.time() - start_time
        
        if found:
            schedule = self.to_schedule()
            schedule = assign_days_balanced(schedule, self.derbies, self.weekend)
            
            print(f"\nSolution found in {elapsed:.2f}s")
            print(f"   Backtracks: {self.backtracks}")
            print(f"   Nodes: {self.nodes}")
            
            if self.dist_matrix:
                travel_dist = total_distance(schedule, self.dist_matrix, self.teams)
                print(f"   Travel distance: {travel_dist:.2f} km")
            
            if is_valid(schedule):
                print(f"   Valid: Yes")
            else:
                violations = csp_validate(schedule)
                print(f"   Valid: No ({len(violations)} violations)")
            
            return schedule
        else:
            print(f"\nNo solution found in {elapsed:.2f}s")
            return None


def assign_days_balanced(schedule: Schedule, derbies: Set[FrozenSet[str]], weekend: List[str]) -> Schedule:
    """Assign days with balanced distribution across all available days."""
    name_to_i: Dict[str, int] = {t.name: i for i, t in enumerate(schedule.teams)}
    nt = len(schedule.teams)
    weekend_set = set(weekend)

    for round_obj in schedule.rounds:
        matches = round_obj.matches
        if not matches:
            continue

        day_choices = round_obj.days
        k = len(matches)
        
        base_per_day = k // len(day_choices)
        remainder = k % len(day_choices)
        
        distribution = [base_per_day + (1 if i < remainder else 0) for i in range(len(day_choices))]
        
        derby_matches = []
        regular_matches = []
        for match in matches:
            if match.pair() in derbies:
                derby_matches.append(match)
            else:
                regular_matches.append(match)
        
        best_assignment = None
        best_score = float('inf')
        
        day_pool = []
        for i, day in enumerate(day_choices):
            day_pool.extend([day] * distribution[i])
        
        for day_perm in set(permutations(day_pool)):
            score = 0
            for i, match in enumerate(matches):
                day = day_perm[i]
                if match.pair() in derbies and day not in weekend_set:
                    score += SOFT_W
            
            if score < best_score:
                best_score = score
                best_assignment = day_perm
        
        if best_assignment:
            for match, day in zip(matches, best_assignment):
                match.day = day
    
    return schedule


def csp_solver(teams, time_limit=60, derbies=None, weekend=None, dist_matrix=None):
    solver = ScheduleCSP(teams, derbies=derbies, weekend=weekend, dist_matrix=dist_matrix)
    return solver.solve(time_limit)

#### TESTING :

In [39]:
print("=" * 80)
print("SPORTS LEAGUE SCHEDULER - MAIN TEST SUITE")
print("=" * 80)

# 1. SETUP

PENALTY = 500

random.seed(42)

data_path = os.path.join(os.getcwd(), 'data')
teams_path = os.path.join(data_path, 'teams.csv')
derbies_path = os.path.join(data_path, 'derbies.csv')

print(f"\nWorking directory: {os.getcwd()}")
print(f"Data path: {data_path}")

# Note : this is for verification . I t will confirm that it accessed the files 

# 2. LOAD DATA

print("\n" + "=" * 60)
print("LOADING DATA")
print("=" * 60)

if not os.path.exists(data_path):
    print(f"\nERROR: Data folder not found at {data_path}")
    print("   Please create a 'data' folder with teams.csv")
    raise SystemExit("Cannot proceed without data files")

if not os.path.exists(teams_path):
    print(f"\nERROR: teams.csv not found at {teams_path}")
    print("   Required columns: team, latitude, longitude")
    raise SystemExit("Cannot proceed without teams.csv")

try:
    teams = load_teams(teams_path)
    distances = build_distance_matrix(teams)
    print(f"\nLoaded {len(teams)} teams from {teams_path}")
    
    print(f"\nTeam list:")
    for i, t in enumerate(teams, 1):
        print(f"   {i:2}. {t.name}: ({t.lat}, {t.lon})")
        
except Exception as e:
    print(f"\nError loading teams: {e}")
    raise

derbies = set()
if os.path.exists(derbies_path):
    try:
        with open(derbies_path, 'r', encoding='latin-1') as f:
            reader = csv.DictReader(f)
            for row in reader:
                derbies.add(frozenset([row['team_a'], row['team_b']]))
        print(f"\nLoaded {len(derbies)} derbies from {derbies_path}")
    except Exception as e:
        print(f"\nWarning: Could not load derbies: {e}")
else:
    print(f"\nNo derbies file found (optional)")

WEEKEND_DAYS = {"Friday", "Saturday"}

print(f"\nSettings:")
print(f"   Weekend days: {WEEKEND_DAYS}")
print(f"   Penalty value: {PENALTY}")

# 3. INITIAL SCHEDULE GENERATION

print("\n" + "=" * 60)
print("TEST 1: INITIAL SCHEDULE GENERATION")
print("=" * 60)

initial_schedule = double_round_robin(teams)
initial_schedule = assign_all_days(initial_schedule)

expected_rounds = 2 * (len(teams) - 1)
expected_matches = len(teams) * (len(teams) - 1)

print(f"\nSchedule generated:")
print(f"   Teams: {len(teams)}")
print(f"   Total rounds: {initial_schedule.total_rounds()} / expected: {expected_rounds}")
print(f"   Total matches: {initial_schedule.total_matches()} / expected: {expected_matches}")

print(f"\nFirst 2 rounds:")
initial_schedule.print_schedule()

# 4. TEST CONSTRAINT VALIDATION

print("\n" + "=" * 60)
print("TEST 2: CONSTRAINT VALIDATION")
print("=" * 60)

v1 = check_one_match_per_round(initial_schedule)
v2 = check_all_pairs_exactly_twice(initial_schedule)
v3 = check_home_away_balance(initial_schedule)
all_valid = is_valid(initial_schedule)

print(f"\nOne match per round: {v1}")
print(f"All pairs exactly twice: {v2}")
print(f"Home/away balanced: {v3}")
print(f"All constraints satisfied: {all_valid}")

# ===========================================================================
# 5. TEST COST FUNCTION
# ============================================================================

print("\n" + "=" * 60)
print("TEST 3: COST FUNCTION")
print("=" * 60)

travel = total_distance(initial_schedule, distances, teams)
rest = Rest_day_penalty(initial_schedule, teams, PENALTY)
derby = Derbies_penalty(initial_schedule, derbies, WEEKEND_DAYS, PENALTY)

away_total = 0
for team in teams:
    away_total += successive_away_penalty(team, initial_schedule, PENALTY)

total_cost_val = cost_function(initial_schedule, teams, distances, derbies, WEEKEND_DAYS, PENALTY)

print(f"\nInitial Schedule Cost Breakdown:")
print(f"   Travel distance: {travel:,.2f} km")
print(f"   Rest day penalty: {rest:,}")
print(f"   Derby penalty: {derby:,}")
print(f"   Consecutive away penalty: {away_total:,}")
print(f"   ---------------------------------")
print(f"   TOTAL COST: {total_cost_val:,.2f}")

# ============================================================================
# 6. TEST NEIGHBORHOOD OPERATORS
# ============================================================================

print("\n" + "=" * 60)
print("TEST 4: NEIGHBORHOOD OPERATORS")
print("=" * 60)

n1 = swap_rounds(initial_schedule, 0, 5)
print(f"\nswap_rounds(0,5): Working")

n2 = swap_home_away(initial_schedule, 0, 0)
print(f"swap_home_away(round0,match0): Working")


n3 = swap_days_in_round(initial_schedule, 0, 0, 1)
print(f"swap_days_in_round(round0,match0,match1): Working")

n4 = swap_third_day(initial_schedule, 0, 1)
print(f"swap_third_day(0,1): Working")

rand_neighbor = generate_one_random_neighbor(initial_schedule)
print(f"generate_one_random_neighbor(): Working")

# ============================================================================
# 7. TEST GREEDY SEARCH
# ============================================================================

print("\n" + "=" * 60)
print("TEST 5: GREEDY SEARCH")
print("=" * 60)

try:
    random.seed(42)
    start = time.time()
    greedy_best, greedy_cost, greedy_history = greedy_schedule(
        initial_schedule, derbies=derbies, weekends=WEEKEND_DAYS,
        teams=teams, dist_matrix=distances, verbose=False
    )
    greedy_elapsed = time.time() - start
    greedy_imp = ((total_cost_val - greedy_cost) / total_cost_val) * 100
    
    print(f"\nGreedy Results:")
    print(f"   Initial cost: {total_cost_val:,.2f}")
    print(f"   Greedy cost: {greedy_cost:,.2f}")
    print(f"   Improvement: {greedy_imp:.1f}%")
    print(f"   Time: {greedy_elapsed:.2f} seconds")
except Exception as e:
    print(f"\nGreedy error: {e}")
    greedy_cost = None
    greedy_elapsed = None

# ============================================================================
# 8. TEST HILL CLIMBING
# ============================================================================

print("\n" + "=" * 60)
print("TEST 6: HILL CLIMBING")
print("=" * 60)

try:
    random.seed(42)
    start = time.time()
    hc_best, hc_cost, hc_history = hill_climbing(
        schedule=initial_schedule, distances=distances, derbies=derbies,
        weekends=WEEKEND_DAYS, teams=teams, max_restarts=3,
        max_iterations=200, max_sideways=20, verbose=False
    )
    hc_elapsed = time.time() - start
    hc_imp = ((total_cost_val - hc_cost) / total_cost_val) * 100
    
    print(f"\nHill Climbing Results:")
    print(f"   Initial cost: {total_cost_val:,.2f}")
    print(f"   HC cost: {hc_cost:,.2f}")
    print(f"   Improvement: {hc_imp:.1f}%")
    print(f"   Time: {hc_elapsed:.2f} seconds")
except Exception as e:
    print(f"\nHill Climbing error: {e}")
    hc_cost = None
    hc_elapsed = None

# ============================================================================
# 9. TEST SIMULATED ANNEALING
# ============================================================================

print("\n" + "=" * 60)
print("TEST 7: SIMULATED ANNEALING")
print("=" * 60)

try:
    random.seed(42)
    start = time.time()
    sa_best, sa_cost, sa_history = simulated_annealing(
        schedule=initial_schedule, distances=distances, derbies=derbies,
        weekends=WEEKEND_DAYS, teams=teams, T_start=8000, T_min=0.1,
        alpha=0.9997, max_iteration=30000, reheat_every=4000,
        reheat_factor=2.0, cost_fn=cost_function
    )
    sa_elapsed = time.time() - start
    sa_imp = ((total_cost_val - sa_cost) / total_cost_val) * 100
    
    print(f"\nSimulated Annealing Results:")
    print(f"   Initial cost: {total_cost_val:,.2f}")
    print(f"   SA cost: {sa_cost:,.2f}")
    print(f"   Improvement: {sa_imp:.1f}%")
    print(f"   Time: {sa_elapsed:.2f} seconds")
except Exception as e:
    print(f"\nSimulated Annealing error: {e}")
    sa_cost = None
    sa_elapsed = None

# ============================================================================
# 10. CSP SOLVER FOR MULTIPLE SIZES (10, 12, 14, 16, 18, 20)
# ============================================================================

print("\n" + "=" * 60)
print("TEST 8: CSP SOLVER FOR MULTIPLE TEAM SIZES")
print("=" * 60)

def create_synthetic_teams(n, base_name="Synthetic"):
    """Create synthetic teams for sizes beyond available data"""
    synthetic_teams = []
    for i in range(n):
        synthetic_teams.append(Team(
            name=f"{base_name}_{i+1}",
            lat=36.35 + (i % 7) * 0.15,
            lon=3.08 + (i // 7) * 0.12
        ))
    return synthetic_teams

team_sizes = [10, 12, 14, 16, 18, 20]
csp_results = []

for n_teams in team_sizes:
    print(f"\n" + "-" * 50)
    print(f"Testing CSP with {n_teams} teams")
    print("-" * 50)
    
    try:
        if n_teams <= len(teams):
            csp_teams = teams[:n_teams]
            print(f"   Using real teams ({n_teams} from CSV)")
        else:
            csp_teams = create_synthetic_teams(n_teams)
            print(f"   Using synthetic teams ({n_teams} teams)")
        
        csp_dist = build_distance_matrix(csp_teams)
        
        csp_team_names = {t.name for t in csp_teams}
        csp_derbies = {d for d in derbies if d <= csp_team_names}
        
        print(f"   Derbies in subset: {len(csp_derbies)}")
        print(f"   Rounds: {2 * (len(csp_teams) - 1)}")
        print(f"   Matches per round: {len(csp_teams) // 2}")
        print(f"   Time limit: 60 seconds")
        
        print(f"   Running CSP solver...")
        start_time = time.time()
        
        csp_result = csp_solver(
            teams=csp_teams,
            time_limit=600,
            derbies=csp_derbies,
            weekend=list(WEEKEND_DAYS),
            dist_matrix=csp_dist
        )
        
        elapsed = time.time() - start_time
        
        if csp_result is not None:
            csp_travel = total_distance(csp_result, csp_dist, csp_teams)
            csp_total = cost_function(csp_result, csp_teams, csp_dist, csp_derbies, WEEKEND_DAYS, PENALTY)
            
            print(f"\n   SUCCESS for {n_teams} teams")
            print(f"      Time: {elapsed:.2f} seconds")
            print(f"      Travel distance: {csp_travel:,.2f} km")
            print(f"      Total cost: {csp_total:,.2f}")
            print(f"      Valid schedule: {is_valid(csp_result)}")
            
            csp_results.append({
                'n': n_teams,
                'status': 'OK',
                'time': elapsed,
                'travel': csp_travel,
                'cost': csp_total
            })
        else:
            print(f"\n   FAILED for {n_teams} teams (no solution within 60 seconds)")
            csp_results.append({
                'n': n_teams,
                'status': 'FAILED',
                'time': elapsed,
                'travel': None,
                'cost': None
            })
            
    except Exception as e:
        print(f"\n   ERROR for {n_teams} teams: {type(e).__name__}: {e}")
        csp_results.append({
            'n': n_teams,
            'status': 'ERROR',
            'time': 0,
            'travel': None,
            'cost': None
        })

# Print CSP summary table
print("\n" + "=" * 60)
print("CSP SOLVER SUMMARY")
print("=" * 60)
print(f"\n{'Teams':<8} {'Status':<12} {'Time (s)':<12} {'Travel (km)':<18} {'Total Cost':<15}")
print("-" * 65)
for r in csp_results:
    if r['status'] == 'OK':
        print(f"{r['n']:<8} {r['status']:<12} {r['time']:<12.2f} {r['travel']:<18,.2f} {r['cost']:<15,.2f}")
    else:
        print(f"{r['n']:<8} {r['status']:<12} {r['time']:<12.2f} {'N/A':<18} {'N/A':<15}")

# ============================================================================
# 11. FINAL COMPARISON
# ============================================================================

print("\n" + "=" * 60)
print("FINAL COMPARISON")
print("=" * 60)

print(f"\n{'Algorithm':<20} {'Final Cost':>15} {'Improvement':>12} {'Time (s)':>10}")
print("-" * 60)
print(f"{'Initial':<20} {total_cost_val:>15,.2f} {'---':>12} {'---':>10}")

results = []

if greedy_cost is not None:
    imp = ((total_cost_val - greedy_cost) / total_cost_val) * 100
    print(f"{'Greedy':<20} {greedy_cost:>15,.2f} {imp:>11.1f}% {greedy_elapsed:>9.2f}")
    results.append(("Greedy", greedy_cost))

if hc_cost is not None:
    imp = ((total_cost_val - hc_cost) / total_cost_val) * 100
    print(f"{'Hill Climbing':<20} {hc_cost:>15,.2f} {imp:>11.1f}% {hc_elapsed:>9.2f}")
    results.append(("Hill Climbing", hc_cost))

if sa_cost is not None:
    imp = ((total_cost_val - sa_cost) / total_cost_val) * 100
    print(f"{'Simulated Annealing':<20} {sa_cost:>15,.2f} {imp:>11.1f}% {sa_elapsed:>9.2f}")
    results.append(("Simulated Annealing", sa_cost))

if results:
    best = min(results, key=lambda x: x[1])
    print(f"\n{'='*60}")
    print(f"BEST ALGORITHM: {best[0]} with cost {best[1]:,.2f}")
    print(f"   Improvement: {((total_cost_val - best[1]) / total_cost_val * 100):.1f}%")

# ============================================================================
# 12. SUMMARY
# ============================================================================

print("\n" + "=" * 60)
print("PROJECT SUMMARY")
print("=" * 60)

print(f"\nConfiguration:")
print(f"   Teams: {len(teams)}")
print(f"   Rounds: {initial_schedule.total_rounds()}")
print(f"   Matches: {initial_schedule.total_matches()}")
print(f"   Derbies: {len(derbies)}")

print(f"\nTechniques:")
print(f"   Circle Method (initial schedule)")
print(f"   4 Neighborhood Operators")
print(f"   Greedy Search")
print(f"   Hill Climbing with Tabu")
print(f"   Simulated Annealing with Reheat")
print(f"   CSP Solver (tested on 10, 12, 14, 16, 18, 20 teams)")

print(f"\nBest Result:")
if results:
    print(f"   Algorithm: {best[0]}")
    print(f"   Final cost: {best[1]:,.2f}")
    print(f"   Improvement: {((total_cost_val - best[1]) / total_cost_val * 100):.1f}%")

print("\n" + "=" * 60)
print("ALL TESTS COMPLETED")
print("=" * 60)

SPORTS LEAGUE SCHEDULER - MAIN TEST SUITE

Working directory: c:\ENSIA 2nd YEAR\S4\ai_project\AI-project-
Data path: c:\ENSIA 2nd YEAR\S4\ai_project\AI-project-\data

LOADING DATA

Loaded 16 teams from c:\ENSIA 2nd YEAR\S4\ai_project\AI-project-\data\teams.csv

Team list:
    1. MC Alger: (36.6817772, 2.961617)
    2. USM Alger: (36.759852, 2.995234)
    3. CR Belouizdad: (36.7042, 3.151)
    4. JS Kabylie: (36.711, 4.0459)
    5. ES Setif: (36.1911, 5.4137)
    6. CS Constantine: (36.365, 6.6147)
    7. MC Oran: (35.7214, -0.5975)
    8. ES Ben Aknoun: (36.74497, 3.07846)
    9. Paradou AC: (36.71509, 3.2244)
   10. ASO Chlef: (36.16167, 1.3088)
   11. JS Saoura: (31.60591, -2.21828)
   12. ES Mostaganem: (35.92453, 0.10537)
   13. MB Rouissat: (31.94154, 5.3421)
   14. Olympique Akbou: (36.74949, 5.04638)
   15. MC El Bayadh: (33.6782, 1.0212)
   16. USM Khenchela: (35.42851, 7.14647)

Loaded 8 derbies from c:\ENSIA 2nd YEAR\S4\ai_project\AI-project-\data\derbies.csv

Settings:
   We